## Simultaneous target analysis of emission of WL-PSI and FRL-PSI of SCo7335, CF9212, FT7521, CT7203, and of TA FT7521

### Defining datasets and inspect data

In [ ]:
from glotaran.io import load_scheme
from pyglotaran_extras.compat import convert


def _case_study_convert(native_result, scheme):
    """Project native v0.8 results for legacy plotting while retaining native results."""
    import numpy as np
    import xarray as xr

    compat_result = convert(native_result)
    for dataset_label, dataset in compat_result.data.items():
        if "irf_center" in dataset.coords:
            irf_center = dataset.coords["irf_center"]
            if irf_center.ndim and np.allclose(irf_center, irf_center.values.flat[0]):
                dataset = dataset.drop_vars("irf_center").assign_coords(
                    irf_center=float(irf_center.values.flat[0])
                )
                compat_result.data[dataset_label] = dataset
        optimization_result = native_result.optimization_results[dataset_label]
        global_dimension = optimization_result.meta.global_dimension
        model_dimension = optimization_result.meta.model_dimension
        input_data = optimization_result.input_data
        residual = optimization_result.residuals
        if isinstance(input_data, xr.Dataset):
            input_data = input_data["data"]
        if isinstance(residual, xr.Dataset):
            residual = residual["residual"]
        fitted_data = input_data - residual
        if {"time", "spectral"}.issubset(fitted_data.dims):
            fitted_data = fitted_data.transpose("time", "spectral")
        dataset["fitted_data"] = fitted_data
        data_model = next(
            experiment.datasets[dataset_label]
            for experiment in scheme.experiments.values()
            if dataset_label in experiment.datasets
        )
        weight = xr.ones_like(residual)
        for weight_item in data_model.weights:
            selected = xr.ones_like(residual, dtype=bool)
            if weight_item.global_interval is not None:
                lower, upper = weight_item.global_interval
                selected = selected & (
                    (residual.coords[global_dimension] >= lower)
                    & (residual.coords[global_dimension] <= upper)
                )
            if weight_item.model_interval is not None:
                lower, upper = weight_item.model_interval
                selected = selected & (
                    (residual.coords[model_dimension] >= lower)
                    & (residual.coords[model_dimension] <= upper)
                )
            weight = weight * xr.where(selected, float(weight_item.value), 1.0)
        dataset["weight"] = weight
        dataset["weighted_residual"] = residual * weight
        dataset["clp"] = optimization_result.fit_decomposition.clp.rename(
            amplitude_label="clp_label"
        )
        dataset["matrix"] = optimization_result.fit_decomposition.matrix.rename(
            amplitude_label="clp_label"
        )
        kinetic_elements = [
            element
            for element in optimization_result.elements.values()
            if "compartment" in element.coords
        ]
        if kinetic_elements:
            species_concentration = xr.concat(
                [
                    element["concentrations"].rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            )
            species_concentration = species_concentration.isel(
                species=~species_concentration.get_index("species").duplicated()
            )
            concentration_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_concentration.dims
            ]
            concentration_order.extend(
                dimension
                for dimension in species_concentration.dims
                if dimension not in concentration_order
            )
            dataset["species_concentration"] = species_concentration.transpose(
                *concentration_order
            )
            species_associated_spectra = xr.concat(
                [element["amplitudes"].rename(compartment="species") for element in kinetic_elements],
                dim="species",
            )
            species_associated_spectra = species_associated_spectra.isel(
                species=~species_associated_spectra.get_index("species").duplicated()
            )
            spectra_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_associated_spectra.dims
            ]
            spectra_order.extend(
                dimension
                for dimension in species_associated_spectra.dims
                if dimension not in spectra_order
            )
            dataset["species_associated_spectra"] = species_associated_spectra.transpose(
                *spectra_order
            )
            initial_concentration = xr.concat(
                [
                    element["initial_concentrations"]
                    .isel(activation=0, drop=True)
                    .rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            )
            dataset["initial_concentration"] = initial_concentration.isel(
                species=~initial_concentration.get_index("species").duplicated()
            )
        spectral_elements = [
            element
            for element in optimization_result.elements.values()
            if "shape" in element.coords
        ]
        if spectral_elements:
            species_spectra = xr.concat(
                [
                    element["concentrations"].squeeze(drop=True).rename(shape="species")
                    for element in spectral_elements
                ],
                dim="species",
            )
            spectral_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_spectra.dims
            ]
            spectral_order.extend(
                dimension for dimension in species_spectra.dims if dimension not in spectral_order
            )
            dataset["species_spectra"] = species_spectra.transpose(*spectral_order)
        dataset.attrs["dataset_scale"] = optimization_result.meta.scale
    return compat_result


def _case_study_matrix_markdown(scheme, element_label, compartments=None):
    """Render a symbolic v0.8 kinetic rate map for legacy notebook display cells."""
    import pandas as pd

    element = scheme.library[element_label]
    compartments = list(compartments or element.compartments)
    table = [["" for _ in compartments] for _ in compartments]
    for (to_compartment, from_compartment), rate in element.rates.items():
        if to_compartment in compartments and from_compartment in compartments:
            table[compartments.index(to_compartment)][compartments.index(from_compartment)] = str(rate)
    return pd.DataFrame(table, index=compartments, columns=compartments).to_markdown()


from __future__ import annotations
from glotaran.io import load_parameters
from glotaran.io import save_result
from pyglotaran_extras import plot_data_overview
from pyglotaran_extras import plot_fitted_traces
from pyglotaran_extras import plot_overview
from pyglotaran_extras import select_plot_wavelengths
from pyglotaran_extras.inspect import show_a_matrixes

The code below defined the (groups of) datasets used in the analysis. Only for a single dataset the plot_data_overview is shown to avoid repetition, it is left as an execirse to the reader to inspect the other data.

In [ ]:
CF9212_DATASETS = {'CF9212FRLtr1': 'data/dispcorr/PSI_9212FR_WLtr124av8nobridgetarget_dispcorra.ascii', 'CF9212WLtr1': 'data/dispcorr/PSI_9212FR_WLtr124av8nobridgetarget_dispcorrb.ascii', 'CF9212FRLtr2': 'data/dispcorr/PSI_9212FR_WLtr124av8nobridgetarget_dispcorrc.ascii', 'CF9212WLtr2': 'data/dispcorr/PSI_9212FR_WLtr124av8nobridgetarget_dispcorrd.ascii', 'CF9212FRLtr4': 'data/dispcorr/PSI_9212FR_WLtr124av8nobridgetarget_dispcorre.ascii'}
plot_data_overview(CF9212_DATASETS['CF9212FRLtr1'], nr_of_data_svd_vectors=4, linlog=True, linthresh=30, irf_location=57)

In [ ]:
CT7203_DATASETS = {'CT7203FRLtr1': 'data/dispcorr/PSI_7203FR_WLtr124av8nobridgetarget_dispcorra.ascii', 'CT7203WLtr1': 'data/dispcorr/PSI_7203FR_WLtr124av8nobridgetarget_dispcorrb.ascii', 'CT7203FRLtr2': 'data/dispcorr/PSI_7203FR_WLtr124av8nobridgetarget_dispcorrc.ascii', 'CT7203WLtr2': 'data/dispcorr/PSI_7203FR_WLtr124av8nobridgetarget_dispcorrd.ascii', 'CT7203FRLtr4': 'data/dispcorr/PSI_7203FR_WLtr124av8nobridgetarget_dispcorre.ascii'}

In [ ]:
Syn7335_DATASETS = {'Syn7335FRLtr1': 'data/dispcorr/PSI_7335FR_WLtr124av8nobridgetarget_dispcorra.ascii', 'Syn7335WLtr1': 'data/dispcorr/PSI_7335FR_WLtr124av8nobridgetarget_dispcorrb.ascii', 'Syn7335FRLtr2': 'data/dispcorr/PSI_7335FR_WLtr124av8nobridgetarget_dispcorrc.ascii', 'Syn7335WLtr2': 'data/dispcorr/PSI_7335FR_WLtr124av8nobridgetarget_dispcorrd.ascii', 'Syn7335FRLtr4': 'data/dispcorr/PSI_7335FR_WLtr124av8nobridgetarget_dispcorre.ascii'}

In [ ]:
STREAK_DATASETS = {'FRLtr1': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorraFRtr1.ascii', 'WLtr1': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrbWLtr1.ascii', 'FRLtr2': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrcFRtr2.ascii', 'WLtr2': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrdWLtr2.ascii', 'FRLtr4': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorreFRtr4.ascii'}

In [ ]:
STREAK_CLP_GUIDE_DATASETS = {'BulkSAS': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrf1Bulk.ascii', 'Red1SAS': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrg2Red1.ascii', 'Red2SAS': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrh3Red2.ascii', 'WLRCSAS': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorri4WLRC.ascii', 'Chlf1SAS': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrj8Chlf1.ascii', 'Chlf2SAS': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrk9Chlf2.ascii', 'FRLRCSAS': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrl10FRLRC.ascii', 'Syn7335FRLRCSAS': 'data/dispcorr/PSI_7335FR_WLtr124av8nobridgetarget_dispcorrm.ascii', 'freeChlfSAS': 'data/dispcorr/PSI_7521FR_WLtr124av8nobridgetarget_dispcorrm12freeChlf.ascii'}

In [ ]:
TA_DATASETS = {'FRL670': 'data/1/CherepanovFR_WLtargeta.ascii', 'FRL700': 'data/1/CherepanovFR_WLtargetb.ascii', 'FRL740': 'data/1/CherepanovFR_WLtargetc.ascii', 'WL700': 'data/1/CherepanovFR_WLtargetd.ascii', 'WL720': 'data/1/CherepanovFR_WLtargete.ascii'}

In [ ]:
TA_CLP_GUIDE_DATASETS = {'Red1SADS': 'data/1/CherepanovFR_WLtargetg.ascii', 'Red2SADS': 'data/1/CherepanovFR_WLtargeth.ascii', 'WLRCSADS': 'data/1/CherepanovFR_WLtargeti.ascii', 'WLRP1SADS': 'data/1/CherepanovFR_WLtargetj.ascii', 'Chlf1SADS': 'data/1/CherepanovFR_WLtargetm.ascii', 'Chlf2SADS': 'data/1/CherepanovFR_WLtargetn.ascii', 'FRLRCSADS': 'data/1/CherepanovFR_WLtargeto.ascii', 'FRLRP1SADS': 'data/1/CherepanovFR_WLtargetp.ascii'}

## Defining filenames for the Target Analysis

### Used model and parameters

In [ ]:
target_model_path = 'models/20230517model_PSI_TA_streak_intermediate_weight_v08.yml'

In [ ]:
target_parameters_path = 'models/20230517optimized_parameters.csv'
optimizedparameters = load_parameters(target_parameters_path)

#### Model file

#### Parameters file

### Create scheme and optimize it
Note that the # sign in front of import has to be removed

In [ ]:
target_scheme = load_scheme(target_model_path)
target_scheme_parameters = optimizedparameters
target_scheme_datasets = {**CF9212_DATASETS, **CT7203_DATASETS, **STREAK_DATASETS, **Syn7335_DATASETS, **STREAK_CLP_GUIDE_DATASETS, **TA_DATASETS, **TA_CLP_GUIDE_DATASETS}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=2, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=2, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result = _case_study_convert(target_result_native, target_scheme)

For reference, the final Cost should be
- 3.6336e+03

To save the results of the optimization we can use the `save_result` command.

Because it saves *everything* it consumes about 40MB of disk space per save.

In [ ]:
save_result(result=target_result_native, result_path='results/20230523/result.yaml', allow_overwrite=True)

## Populations and SADS estimated with the sequential scheme

<sub>Note: The color scheme of the plots in this notebook may not match published figures.</sub>

In [ ]:
import matplotlib.pyplot as plt
from pyglotaran_extras.plotting.plot_concentrations import plot_concentrations
from pyglotaran_extras.plotting.plot_spectra import plot_sas

def plot_concentration_and_spectra(result_dataset, linthresh=1):
    (fig, axes) = plt.subplots(1, 2, figsize=(15, 4))
    plot_concentrations(result_dataset, axes[0], center_λ=0, linlog=True, linthresh=linthresh)
    plot_sas(result_dataset, axes[1])
    return (fig, axes)
(fig, axes) = plot_concentration_and_spectra(target_result.data['WL700'])
(fig, axes) = plot_concentration_and_spectra(target_result.data['FRL700'])
axes[0].set_xlabel('Time (ps)')
axes[0].set_ylabel('')
axes[0].axhline(0, color='k', linewidth=1)
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('SADS (mOD)')
axes[1].set_title('SADS')
axes[1].axhline(0, color='k', linewidth=1)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
(fig, axes) = plot_concentration_and_spectra(target_result.data['FRLtr2'], linthresh=10)
(fig, axes) = plot_concentration_and_spectra(target_result.data['CT7203FRLtr2'], linthresh=10)
(fig, axes) = plot_concentration_and_spectra(target_result.data['CF9212FRLtr2'], linthresh=10)
(fig, axes) = plot_concentration_and_spectra(target_result.data['Syn7335FRLtr2'], linthresh=10)

In [ ]:
target_result.root_mean_square_error

In [ ]:
additional_penalties = __import__('numpy').atleast_2d(target_result.additional_penalty)
table = '\n'.join([' | '.join([f'{item:.2f}' for item in row]) for row in additional_penalties])
print(table)

In [ ]:
target_result

In [ ]:
target_result.optimized_parameters

In [ ]:
show_a_matrixes(target_result)

In [ ]:
_case_study_matrix_markdown(target_scheme, 'complexFRL740', ['Ant1', 'Bulk', 'Red1', 'Red2', 'FRLRC', 'FRLRP1', 'FRLRP2', 'Chlf1', 'Chlf2'])

### Result plots

## Comparison of the estimated SAS (orange) and the guidance spectra (blue)
The guidance spectra are shapes derived elsewhere

In [ ]:
target_result.data['Chlf1SAS'].data.plot()
target_result.data['Chlf1SAS'].fitted_data.plot()
target_result.data['Chlf2SAS'].data.plot()
target_result.data['Chlf2SAS'].fitted_data.plot()

In [ ]:
target_result.data['freeChlfSAS'].data.plot()
target_result.data['freeChlfSAS'].fitted_data.plot()

In [ ]:
target_result.data['Red1SAS'].data.plot()
target_result.data['Red1SAS'].fitted_data.plot()

In [ ]:
target_result.data['Red2SAS'].data.plot()
target_result.data['Red2SAS'].fitted_data.plot()

In [ ]:
target_result.data['Red1SAS'].data.plot()
target_result.data['Red1SAS'].fitted_data.plot()
target_result.data['Red2SAS'].data.plot()
target_result.data['Red2SAS'].fitted_data.plot()

In [ ]:
target_result.data['WLRP1SADS'].data.plot()
target_result.data['WLRP1SADS'].fitted_data.plot()

In [ ]:
target_result.data['FRLRP1SADS'].data.plot()
target_result.data['FRLRP1SADS'].fitted_data.plot()

In [ ]:
target_result.data['FRLRCSADS'].data.plot()
target_result.data['FRLRCSADS'].fitted_data.plot()

In [ ]:
target_result.data['FRLRCSAS'].data.plot()
target_result.data['FRLRCSAS'].fitted_data.plot()

In [ ]:
target_result.data['Syn7335FRLRCSAS'].data.plot()
target_result.data['Syn7335FRLRCSAS'].fitted_data.plot()

### Overlay plots of concentrations

In [ ]:
from cycler import cycler
(fig, ax) = plt.subplots(1, 1)
myFRLcolors = ['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'tab:brown', 'm', 'tab:purple']
custom_cycler = cycler(color=myFRLcolors, linestyle=['--'] * 10)
ax.set_prop_cycle(custom_cycler)
concplot = target_result.data['FRL740'].species_concentration.plot.line(x='time', add_legend=False, ax=ax)
custom_cycler = cycler(color=myFRLcolors, linestyle=['-'] * 10)
ax.set_prop_cycle(custom_cycler)
concplot = target_result.data['FRL700'].species_concentration.plot.line(x='time', add_legend=False, ax=ax)
custom_cycler = cycler(color=myFRLcolors, linestyle=[':'] * 10)
ax.set_prop_cycle(custom_cycler)
concplot = target_result.data['FRL670'].species_concentration.plot.line(x='time', add_legend=False, ax=ax)
ax.set_xscale('symlog', linthresh=1)
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Population')
ax.legend(target_result.data['FRL740'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

In [ ]:
from cycler import cycler
from pyglotaran_extras.plotting.utils import shift_time_axis_by_irf_location
(fig, ax) = plt.subplots(1, 1)
myFRLcolors = ['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'tab:brown', 'tab:olive']
custom_cycler = cycler(color=myFRLcolors, linestyle=['-'] * 8)
ax.set_prop_cycle(custom_cycler)
species_concentration = shift_time_axis_by_irf_location(target_result.data['FRLtr2'].species_concentration, target_result.data['FRLtr2'].irf_center)
concplot = species_concentration.plot.line(x='time', add_legend=False, ax=ax)
custom_cycler = cycler(color=myFRLcolors, linestyle=['--'] * 8)
ax.set_prop_cycle(custom_cycler)
species_concentration = shift_time_axis_by_irf_location(target_result.data['WLtr2'].species_concentration, target_result.data['WLtr2'].irf_center)
concplot = species_concentration.plot.line(x='time', add_legend=False, ax=ax)
ax.set_xscale('symlog', linthresh=10)
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Population')
ax.legend(target_result.data['FRLtr2'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

## FRL TA SADS

In [ ]:
(fig, ax) = plt.subplots(1, 1)
myFRLcolors = ['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'tab:brown', 'm', 'tab:purple']
custom_cycler = cycler(color=myFRLcolors)
ax.set_prop_cycle(custom_cycler)
SADSplot = target_result.data['FRL740'].species_associated_spectra.plot.line(x='spectral', add_legend=False, ax=ax)
ax.legend(target_result.data['FRL740'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

## WL TA SADS

In [ ]:
(fig, ax) = plt.subplots(1, 1)
SADSplot = target_result.data['WL720'].species_associated_spectra.plot.line(x='spectral', add_legend=False, ax=ax)
ax.legend(target_result.data['WL720'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

## FRL SAS

In [ ]:
(fig, ax) = plt.subplots(1, 1)
SADSplot = target_result.data['FRLtr2'].species_associated_spectra.plot.line(x='spectral', add_legend=False, ax=ax)
ax.legend(target_result.data['FRLtr2'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

In [ ]:
(fig, ax) = plt.subplots(1, 1)
SADSplot = target_result.data['CF9212FRLtr2'].species_associated_spectra.plot.line(x='spectral', add_legend=False, ax=ax)
ax.legend(target_result.data['CF9212FRLtr2'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

In [ ]:
(fig, ax) = plt.subplots(1, 1)
SADSplot = target_result.data['CT7203FRLtr2'].species_associated_spectra.plot.line(x='spectral', add_legend=False, ax=ax)
ax.legend(target_result.data['CT7203FRLtr2'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

In [ ]:
(fig, ax) = plt.subplots(1, 1)
SADSplot = target_result.data['Syn7335FRLtr2'].species_associated_spectra.plot.line(x='spectral', add_legend=False, ax=ax)
ax.legend(target_result.data['Syn7335FRLtr2'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

## WL SAS

In [ ]:
(fig, ax) = plt.subplots(1, 1)
SADSplot = target_result.data['WLtr2'].species_associated_spectra.plot.line(x='spectral', add_legend=False, ax=ax)
ax.legend(target_result.data['WLtr2'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

In [ ]:
(fig, ax) = plt.subplots(1, 1)
SADSplot = target_result.data['Syn7335WLtr2'].species_associated_spectra.plot.line(x='spectral', add_legend=False, ax=ax)
ax.legend(target_result.data['Syn7335WLtr2'].species.values, bbox_to_anchor=(0.8, -0.13), ncol=3)

## overlay of FRL-RP2 SADS
note that with the different excitation wavelengths the final SADS differs, cf. the EADS in the Cherepanov 2020 paper.

In [ ]:
(fig, ax) = plt.subplots(1, 1)
target_result.data['FRL740'].species_associated_spectra.sel(species=['FRLRP2']).plot.line(x='spectral', label='FRLRP2', ax=ax)
target_result.data['FRL700'].species_associated_spectra.sel(species=['FRLRP2a']).plot.line(x='spectral', label='FRLRP2a', ax=ax)
plot670 = target_result.data['FRL670'].species_associated_spectra.sel(species=['FRLRP2b']).plot.line(x='spectral', label='FRLRP2b', ax=ax)
ax.legend()

## overlay of WL-RP2 SADS
note that with the different excitation wavelengths the final SADS differs, cf. the EADS in the Cherepanov 2020 paper.

In [ ]:
(fig, ax) = plt.subplots(1, 1)
target_result.data['WL720'].species_associated_spectra.sel(species=['WLRP2']).plot.line(x='spectral', label='WLRP2', ax=ax)
target_result.data['WL700'].species_associated_spectra.sel(species=['WLRP2a']).plot.line(x='spectral', label='WLRP2a', ax=ax)
ax.legend()

#### Overview plots per dataset

## TA

In [ ]:
from cycler import cycler
plot_overview(target_result.data['FRL740'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

In [ ]:
plot_overview(target_result.data['FRL700'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

In [ ]:
plot_overview(target_result.data['FRL670'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['WL720'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['WL700'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

## Streak (selection)

In [ ]:
from cycler import cycler
plot_overview(target_result.data['FRLtr2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=30, cycler=cycler(color=['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'y', 'tab:brown']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['FRLtr4'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=30, cycler=cycler(color=['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'y', 'tab:brown']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['WLtr2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=30, cycler=cycler(color=['g', 'tab:orange', 'r', 'tab:gray', 'tab:brown', 'y', 'y']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['CT7203WLtr1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=15, cycler=cycler(color=['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'y', 'tab:brown']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['CT7203WLtr2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=15, cycler=cycler(color=['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'y', 'tab:brown']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['CT7203FRLtr1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=15, cycler=cycler(color=['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'y', 'tab:brown']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['CT7203FRLtr2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=30, cycler=cycler(color=['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'y', 'tab:brown']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['CT7203FRLtr4'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=15, cycler=cycler(color=['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'y', 'tab:brown']))

In [ ]:
from cycler import cycler
plot_overview(target_result.data['CF9212FRLtr4'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=True, linthresh=15, cycler=cycler(color=['g', 'tab:orange', 'r', 'k', 'm', 'tab:purple', 'y', 'tab:brown']))

#### Fit quality
overlays of traces and fits, 16 wavelengths per dataset

In [ ]:
target_result_streak = (target_result.data['FRLtr1'], target_result.data['WLtr1'], target_result.data['FRLtr2'], target_result.data['WLtr2'], target_result.data['FRLtr4'])

In [ ]:
CT7203target_result_streak = (target_result.data['CT7203FRLtr1'], target_result.data['CT7203WLtr1'], target_result.data['CT7203FRLtr2'], target_result.data['CT7203WLtr2'], target_result.data['CT7203FRLtr4'])

In [ ]:
CF9212target_result_streak = (target_result.data['CF9212FRLtr1'], target_result.data['CF9212WLtr1'], target_result.data['CF9212FRLtr2'], target_result.data['CF9212WLtr2'], target_result.data['CF9212FRLtr4'])

In [ ]:
Syn7335target_result_streak = (target_result.data['Syn7335FRLtr1'], target_result.data['Syn7335WLtr1'], target_result.data['Syn7335FRLtr2'], target_result.data['Syn7335WLtr2'], target_result.data['Syn7335FRLtr4'])

In [ ]:
target_result_TA = (target_result.data['FRL670'], target_result.data['FRL700'], target_result.data['FRL740'], target_result.data['WL700'], target_result.data['WL720'])

In [ ]:
wavelengths = select_plot_wavelengths(target_result_streak, equidistant_wavelengths=True)
plot_fitted_traces(target_result_streak, wavelengths, linlog=True, linthresh=15)

In [ ]:
wavelengths = select_plot_wavelengths(CT7203target_result_streak, equidistant_wavelengths=True)
plot_fitted_traces(CT7203target_result_streak, wavelengths, linlog=True, linthresh=15)

In [ ]:
wavelengths = select_plot_wavelengths(target_result_streak, equidistant_wavelengths=True)
plot_fitted_traces(CF9212target_result_streak, wavelengths, linlog=True, linthresh=15)

In [ ]:
wavelengths = select_plot_wavelengths(target_result_streak, equidistant_wavelengths=True)
plot_fitted_traces(Syn7335target_result_streak, wavelengths, linlog=True, linthresh=15)

In [ ]:
wavelengths = select_plot_wavelengths(target_result_TA, equidistant_wavelengths=True)
plot_fitted_traces(target_result_TA, wavelengths, linlog=True, linthresh=1)